# Etapa 1 — Carga e consolidação dos dados

Esta etapa lê os arquivos brutos do conjunto *NBA Database (1947–Present)* (Moore, Kaggle),
aplica os filtros de escopo e produz uma base com **uma linha por equipe em cada partida**,
contendo as estatísticas da própria equipe e as do adversário.

| | Arquivo | Conteúdo |
|---|---|---|
| Entrada | `dados/brutos/Games.csv` | uma linha por partida: data, equipes, placar e tipo de jogo |
| Entrada | `dados/brutos/TeamStatisticsExtended.csv` | uma linha por equipe em cada partida: estatísticas de caixa e de perfil de pontuação |
| Saída | `dados/processados/partidas_times.parquet` | base consolidada, lida pela etapa 2 |
| Saída | `dados/processados/metadados_carga.json` | parâmetros e contagens desta execução |

As temporadas são identificadas pelo **ano em que terminam**: a temporada 2014-15 é a 2015.

## 1. Escopo do estudo

- **Duas eras de dez temporadas.** A Era Clássica vai de 2005 a 2014 e a Era Moderna de 2015 a
  2026, sem as temporadas de 2020 e 2021, disputadas em bolha e com calendário reduzido pela
  pandemia de COVID-19. Com essa exclusão, as duas eras ficam com o mesmo número de temporadas.
- **Só jogos que contam para a temporada regular.** Entram as partidas de temporada regular e as
  da Copa NBA que valem para a classificação da temporada regular (fase de grupos, quartas de
  final e semifinais). A final da Copa não conta para essa classificação e fica de fora. Na
  temporada de 2024 essas partidas vêm em `Games.csv` com o tipo `NBA Emirates Cup`; nas
  temporadas seguintes já vêm como `Regular Season`.

In [1]:
from datetime import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

DIR_BRUTO = Path("dados/brutos")
DIR_SAIDA = Path("dados/processados")

ERAS = {"Classica": range(2005, 2015),     # 2004-05 a 2013-14
        "Moderna": range(2015, 2027)}      # 2014-15 a 2025-26
TEMPORADAS_EXCLUIDAS = {2020, 2021}
TIPOS_JOGO_VALIDOS = {"Regular Season", "NBA Emirates Cup"}

for era, anos in ERAS.items():
    temporadas = [t for t in anos if t not in TEMPORADAS_EXCLUIDAS]
    print(f"{era}: {len(temporadas)} temporadas ({temporadas[0]} a {temporadas[-1]})")
print(f"tipos de jogo aceitos: {sorted(TIPOS_JOGO_VALIDOS)}")

Classica: 10 temporadas (2005 a 2014)
Moderna: 10 temporadas (2015 a 2026)
tipos de jogo aceitos: ['NBA Emirates Cup', 'Regular Season']


## 2. Variáveis lidas das estatísticas por equipe

São lidas três classes de colunas, renomeadas para os nomes curtos usados no restante do
pipeline:

1. **Box score**: pontos, arremessos de quadra, de três pontos e
   lances livres (convertidos e tentados), rebotes ofensivos e defensivos, assistências, roubos
   de bola, bloqueios, erros com perda de posse, faltas e minutos jogados.
2. **Contagens de tipos de pontuação**: pontos no garrafão, em contra-ataque, de segunda
   chance, após erro do adversário e do banco de reservas.
3. **Percentuais sem contagem equivalente**: a fração dos pontos que veio de arremessos de meia
   distância e a fração das cestas de três pontos que foram assistidas.

Os percentuais exigem cuidado. A etapa 2 resume cada equipe em janelas de vários jogos, e a
média dos percentuais de vários jogos não é o percentual do conjunto: um jogo com 2 cestas de
três pesaria tanto quanto um com 20. Por isso cada percentual é convertido de volta no
**numerador** que o originou (percentual × volume do jogo). Na janela somam-se numeradores e
volumes e divide-se no fim, de modo que cada jogo pesa pelo seu volume.

As estatísticas do adversário não são lidas do arquivo: saem da linha da outra equipe na mesma
partida (seção 5), o que garante que os dois lados de um jogo sejam sempre consistentes.

In [2]:
MAPA_STATS = {
    "teamScore": "pts",
    "fieldGoalsMade": "fgm",        "fieldGoalsAttempted": "fga",
    "threePointersMade": "fg3m",    "threePointersAttempted": "fg3a",
    "freeThrowsMade": "ftm",        "freeThrowsAttempted": "fta",
    "reboundsOffensive": "orb",     "reboundsDefensive": "drb",
    "assists": "ast",               "steals": "stl",
    "blocks": "blk",                "turnovers": "tov",
    "foulsPersonal": "pf",          "numMinutes": "minutos",
}
MAPA_PERFIL = {
    "pointsInThePaint": "pts_garrafao",
    "pointsFastBreak": "pts_contra_ataque",
    "pointsSecondChance": "pts_segunda_chance",
    "pointsOffTurnovers": "pts_erro_adv",
    "benchPoints": "pts_banco",
}
# percentual de origem -> (numerador reconstruído, volume que multiplica o percentual)
MAPA_PONDERADOS = {
    "percentPoints2PointMidRange": ("num_meia_distancia", "pts"),
    "percentAssisted3PointMade": ("num_3p_assistidos", "fg3m"),
}
COLS_STATS = [*MAPA_STATS.values(), *MAPA_PERFIL.values(),
              *(numerador for numerador, _ in MAPA_PONDERADOS.values())]

print(f"{len(COLS_STATS)} estatísticas por equipe:")
print(COLS_STATS)

22 estatísticas por equipe:
['pts', 'fgm', 'fga', 'fg3m', 'fg3a', 'ftm', 'fta', 'orb', 'drb', 'ast', 'stl', 'blk', 'tov', 'pf', 'minutos', 'pts_garrafao', 'pts_contra_ataque', 'pts_segunda_chance', 'pts_erro_adv', 'pts_banco', 'num_meia_distancia', 'num_3p_assistidos']


## 3. Partidas e filtros de escopo

Cada partida recebe a temporada (jogos de setembro em diante pertencem à temporada que termina
no ano seguinte) e a era. A primeira tabela mostra quantas partidas restam após cada filtro; a
segunda, os tipos de jogo presentes nas temporadas do escopo e quais deles entram.

In [3]:
g = pd.read_csv(DIR_BRUTO / "Games.csv", low_memory=False)
jogos = pd.DataFrame({
    "game_id": g["gameId"].astype("int64"),
    # o horário vem no fuso da costa leste dos EUA, que define a data oficial do jogo
    "data": pd.to_datetime(g["gameDateTimeEst"], format="%Y-%m-%d %H:%M:%S"),
    "tipo": g["gameType"].astype(str).str.strip(),
    "home_id": g["hometeamId"].astype("int64"),
    "away_id": g["awayteamId"].astype("int64"),
    "home_pts": pd.to_numeric(g["homeScore"], errors="coerce"),
    "away_pts": pd.to_numeric(g["awayScore"], errors="coerce"),
})
jogos["temporada"] = np.where(jogos["data"].dt.month >= 9,
                              jogos["data"].dt.year + 1, jogos["data"].dt.year)
jogos["era"] = jogos["temporada"].map({t: era for era, anos in ERAS.items() for t in anos})

tipo_ok = jogos["tipo"].isin(TIPOS_JOGO_VALIDOS)
no_recorte = jogos["era"].notna()
fora_pandemia = ~jogos["temporada"].isin(TEMPORADAS_EXCLUIDAS)
funil = {
    "todas as partidas do arquivo": len(jogos),
    "temporada regular e Copa NBA (sem a final)": int(tipo_ok.sum()),
    "temporadas de 2005 a 2026": int((tipo_ok & no_recorte).sum()),
    "sem as temporadas de 2020 e 2021": int((tipo_ok & no_recorte & fora_pandemia).sum()),
}
print(pd.Series(funil, name="partidas").to_string())

escopo = no_recorte & fora_pandemia
tipos = (jogos[escopo].groupby("tipo")
         .agg(partidas=("game_id", "size"),
              primeira=("temporada", "min"), ultima=("temporada", "max")))
tipos["entra"] = np.where(tipos.index.isin(TIPOS_JOGO_VALIDOS), "sim", "não")
print("\nTipos de jogo nas temporadas do escopo:")
print(tipos.sort_values("partidas", ascending=False).to_string())

jogos = jogos[tipo_ok & escopo].copy()

todas as partidas do arquivo                  73279
temporada regular e Copa NBA (sem a final)    66834
temporadas de 2005 a 2026                     26494
sem as temporadas de 2020 e 2021              24355

Tipos de jogo nas temporadas do escopo:
                    partidas  primeira  ultima entra
tipo                                                
Regular Season         24289      2005    2026   sim
Preseason               1711      2005    2026   não
Playoffs                1676      2005    2026   não
NBA Emirates Cup          66      2024    2024   sim
Play-in Tournament        30      2022    2026   não
All-Star Game              7      2026    2026   não
NBA Cup                    1      2024    2024   não
Emirates NBA Cup           1      2026    2026   não


Três verificações de integridade sobre as partidas que ficaram: placar ausente, identificador
repetido e placar empatado. Como não há empate na NBA, um placar igual é registro inválido da
base (0 a 0 em jogo não disputado ou placar final incorreto) e a partida é removida. Em
seguida é criada a variável resposta, `vitoria_mandante`, igual a 1 quando o time da casa vence.

In [4]:
sem_placar = jogos["home_pts"].isna() | jogos["away_pts"].isna()
duplicado = jogos["game_id"].duplicated()
empate = jogos["home_pts"] == jogos["away_pts"]
print(f"sem placar: {sem_placar.sum()} | identificador repetido: {duplicado.sum()} | "
      f"placar empatado: {empate.sum()}")
print(jogos.loc[empate, ["game_id", "data", "temporada", "home_pts", "away_pts"]]
      .to_string(index=False))
jogos = jogos[~(sem_placar | duplicado | empate)]

jogos["vitoria_mandante"] = (jogos["home_pts"] > jogos["away_pts"]).astype("int8")
jogos = jogos.sort_values(["data", "game_id"]).reset_index(drop=True)
print(f"{len(jogos):,} partidas no escopo | o mandante vence "
      f"{jogos['vitoria_mandante'].mean():.1%} delas")

sem placar: 0 | identificador repetido: 0 | placar empatado: 4
 game_id                data  temporada  home_pts  away_pts
22400988 2025-04-01 20:00:00       2025         0         0
22400627 2025-03-19 21:00:00       2025         0         0
21000028 2010-10-29 20:30:00       2011        90        90
20400090 2004-11-13 22:00:00       2005       105       105
24,351 partidas no escopo | o mandante vence 58.3% delas


## 4. Estatísticas por equipe

Lê só as colunas necessárias, restritas às partidas do escopo, e reconstrói os numeradores dos
percentuais. Também lê a marcação oficial de prorrogação, `otAllPoints` (pontos marcados no
tempo extra).

A prorrogação **não é variável preditora**: só se conhece depois que o jogo termina. Ela é
guardada para uma análise de controle na etapa 3, que mede o desempenho do modelo separadamente
nos jogos decididos no tempo extra.

In [5]:
colunas = ["gameId", "teamId", *MAPA_STATS, *MAPA_PERFIL, *MAPA_PONDERADOS, "otAllPoints"]
stats = pd.read_csv(DIR_BRUTO / "TeamStatisticsExtended.csv", usecols=colunas,
                    low_memory=False)
stats = stats[stats["gameId"].isin(jogos["game_id"])]
stats = stats.rename(columns={"gameId": "game_id", "teamId": "team_id",
                              **MAPA_STATS, **MAPA_PERFIL})

for origem, (numerador, volume) in MAPA_PONDERADOS.items():
    stats[numerador] = stats[origem] * stats[volume]

# Partida sem marcação oficial fica com NaN e recebe a regra dos minutos na seção 5.
ot = pd.to_numeric(stats["otAllPoints"], errors="coerce")
stats["prorrogacao"] = np.where(ot.notna(), (ot > 0).astype("float32"), np.nan)

stats[["game_id", "team_id"]] = stats[["game_id", "team_id"]].astype("int64")
stats[COLS_STATS] = stats[COLS_STATS].apply(pd.to_numeric, errors="coerce").astype("float32")
repetidas = stats.duplicated(["game_id", "team_id"])
stats = stats.loc[~repetidas, ["game_id", "team_id", *COLS_STATS, "prorrogacao"]]
print(f"{len(stats):,} linhas equipe-partida | {repetidas.sum()} repetidas removidas")

48,700 linhas equipe-partida | 0 repetidas removidas


## 5. Base consolidada: equipe e adversário na mesma linha

Junta as estatísticas às partidas e aplica três verificações de consistência:

1. cada partida precisa ter exatamente duas linhas, uma do mandante e uma do visitante;
2. os pontos das estatísticas precisam coincidir com o placar oficial (havendo divergência,
   prevalece o placar de `Games.csv`);
3. a linha de cada equipe recebe as estatísticas do adversário, com o prefixo `opp_`. Elas são
   necessárias para as métricas que dependem dos dois lados da quadra, como a eficiência
   defensiva e os percentuais de rebote.

In [6]:
base = stats.merge(jogos, on="game_id", how="inner")
participa = (base["team_id"] == base["home_id"]) | (base["team_id"] == base["away_id"])
base = base[participa].copy()
base["mandante"] = (base["team_id"] == base["home_id"]).astype("int8")

contagem = base.groupby("game_id")["mandante"].agg(["size", "sum"])
incompletas = contagem.index[(contagem["size"] != 2) | (contagem["sum"] != 1)]
sem_stats = set(jogos["game_id"]) - set(base["game_id"])
base = base[~base["game_id"].isin(incompletas)]

esperado = np.where(base["mandante"] == 1, base["home_pts"], base["away_pts"])
divergentes = int((base["pts"].notna() & (base["pts"] != esperado)).sum())
base["pts"] = esperado.astype("float32")

adv = base[["game_id", "team_id", *COLS_STATS]].rename(
    columns={"team_id": "opp_team_id", **{c: f"opp_{c}" for c in COLS_STATS}})
base = base.merge(adv, on="game_id")
base = base[base["team_id"] != base["opp_team_id"]].copy()
base["vitoria"] = np.where(base["mandante"] == 1, base["vitoria_mandante"],
                           1 - base["vitoria_mandante"]).astype("int8")

print(f"linhas de equipe que não disputou a partida: {(~participa).sum()}")
print(f"partidas sem as duas linhas: {len(incompletas)} | sem estatísticas: {len(sem_stats)}")
print(f"linhas com pontos diferentes do placar oficial: {divergentes}")
print(f"{base['game_id'].nunique():,} partidas consolidadas")

linhas de equipe que não disputou a partida: 0
partidas sem as duas linhas: 0 | sem estatísticas: 1
linhas com pontos diferentes do placar oficial: 0
24,350 partidas consolidadas


### Prorrogação

A marcação oficial (`otAllPoints`) só vem preenchida nos jogos que tiveram prorrogação, e falta
inteiramente na temporada de 2022. Onde ela está vazia, decide o tempo total de quadra: um jogo
sem prorrogação soma 240 minutos por equipe (5 jogadores × 48 minutos), e cada prorrogação
acrescenta 25. O limiar de 242 minutos absorve pequenas imprecisões do registro. A tabela
mostra, por temporada, quantos jogos com prorrogação foram identificados por cada critério.

In [7]:
oficial = base.groupby("game_id")["prorrogacao"].transform("max")
minutos = base.groupby("game_id")["minutos"].transform("max")
pelos_minutos = np.where(minutos.isna(), np.nan, (minutos > 242).astype(float))
base["prorrogacao"] = np.where(oficial.notna(), oficial, pelos_minutos)

criterio = pd.Series(np.where(oficial.notna(), "marcação oficial", "regra dos minutos"),
                     index=base.index)
prorrogadas = base[(base["mandante"] == 1) & (base["prorrogacao"] == 1)]
print("Jogos com prorrogação, por temporada e critério de identificação:")
print(pd.crosstab(prorrogadas["temporada"], criterio[prorrogadas.index], colnames=["critério"],
                  margins=True, margins_name="total").to_string())

base = base.drop(columns=["home_id", "away_id", "home_pts", "away_pts", "tipo"])
frente = ["game_id", "data", "temporada", "era", "team_id", "opp_team_id",
          "mandante", "vitoria", "vitoria_mandante", "prorrogacao"]
base = base[frente + [c for c in base.columns if c not in frente]]
base = (base.sort_values(["data", "game_id", "mandante"], ascending=[True, True, False])
            .reset_index(drop=True))
print(f"\nbase final: {len(base):,} linhas ({base['game_id'].nunique():,} partidas × 2 equipes)"
      f" e {base.shape[1]} colunas")

Jogos com prorrogação, por temporada e critério de identificação:
critério   marcação oficial  regra dos minutos  total
temporada                                            
2005                     79                  0     79
2006                     86                  0     86
2007                     88                  0     88
2008                     64                  0     64
2009                     70                  0     70
2010                     72                  0     72
2011                     81                  0     81
2012                     64                  0     64
2013                     75                  0     75
2014                     79                  0     79
2015                     76                  0     76
2016                     77                  0     77
2017                     70                  0     70
2018                     61                  0     61
2019                     67                  0     67
2022            

## 6. Resumo por temporada

Conferência da base: número de partidas (o esperado é 1.230 por temporada; a de 2012 teve 990
por causa da paralisação patronal, o *lockout*, de 2011), taxa de vitória do mandante e
proporção de jogos com prorrogação. As duas últimas colunas descrevem a mudança de estilo
estudada no trabalho:

- **3PAr** (*three-point attempt rate*) = tentativas de três pontos ÷ tentativas de arremesso
  de quadra, a fração dos arremessos que é de três;
- **3P%** = cestas de três ÷ tentativas de três, o aproveitamento nesses arremessos.

In [8]:
mandantes = base[base["mandante"] == 1]
resumo = mandantes.groupby(["era", "temporada"]).agg(
    partidas=("game_id", "size"), vitoria_mandante=("vitoria", "mean"),
    prorrogacao=("prorrogacao", "mean"))
somas = base.groupby(["era", "temporada"])[["fg3m", "fg3a", "fga"]].sum()
resumo["3PAr"] = somas["fg3a"] / somas["fga"]
resumo["3P%"] = somas["fg3m"] / somas["fg3a"]
print(resumo.round(3).to_string())

somas_era = base.groupby("era")[["fg3m", "fg3a", "fga"]].sum()
print("\nPor era:")
print(pd.DataFrame({"partidas": mandantes.groupby("era").size(),
                    "vitoria_mandante": mandantes.groupby("era")["vitoria"].mean(),
                    "3PAr": somas_era["fg3a"] / somas_era["fga"],
                    "3P%": somas_era["fg3m"] / somas_era["fg3a"]}).round(3).to_string())

                    partidas  vitoria_mandante  prorrogacao   3PAr    3P%
era      temporada                                                       
Classica 2005           1229             0.605        0.064  0.196  0.355
         2006           1230             0.603        0.070  0.202  0.358
         2007           1230             0.591        0.072  0.213  0.358
         2008           1230             0.601        0.052  0.222  0.362
         2009           1230             0.608        0.057  0.224  0.367
         2010           1230             0.594        0.059  0.222  0.355
         2011           1229             0.605        0.066  0.222  0.359
         2012            990             0.586        0.065  0.226  0.349
         2013           1229             0.612        0.061  0.243  0.359
         2014           1230             0.580        0.064  0.259  0.360
Moderna  2015           1230             0.575        0.062  0.268  0.350
         2016           1230          

## 7. Cobertura das colunas por temporada

Uma coluna ausente numa temporada inteira distorceria a comparação entre as eras. A lista
abaixo mostra as temporadas com mais de 1% de valores ausentes em alguma coluna; a etapa 2
descarta as variáveis afetadas.

In [9]:
ausentes = base.groupby("temporada")[COLS_STATS].apply(lambda t: t.isna().mean())
problemas = ausentes[(ausentes > 0.01).any(axis=1)]
if problemas.empty:
    print("nenhuma temporada com mais de 1% de ausentes")
for temporada, linha in problemas.iterrows():
    piores = linha[linha > 0.01].sort_values(ascending=False)
    print(f"{temporada}: " + ", ".join(f"{c} {v:.0%}" for c, v in piores.items()))

2022: pts_garrafao 100%, pts_contra_ataque 100%, pts_segunda_chance 100%, pts_banco 100%


## 8. Salvar

In [10]:
DIR_SAIDA.mkdir(parents=True, exist_ok=True)
arquivo = DIR_SAIDA / "partidas_times.parquet"
base.to_parquet(arquivo, index=False)

funil["após as verificações de integridade"] = int(base["game_id"].nunique())
metadados = {
    "executado_em": datetime.now().isoformat(timespec="seconds"),
    "fonte_estatisticas": "TeamStatisticsExtended.csv",
    "origem_prorrogacao": "otAllPoints; na ausência, numMinutes > 242",
    "eras": {e: [min(a), max(a)] for e, a in ERAS.items()},
    "temporadas_excluidas": sorted(TEMPORADAS_EXCLUIDAS),
    "tipos_jogo": sorted(TIPOS_JOGO_VALIDOS),
    "funil_partidas": funil,
    "n_linhas": int(len(base)),
    "n_jogos": int(base["game_id"].nunique()),
    "jogos_por_temporada": mandantes.groupby("temporada")["game_id"].nunique().to_dict(),
}
(DIR_SAIDA / "metadados_carga.json").write_text(
    json.dumps(metadados, indent=2, ensure_ascii=False, default=int), encoding="utf-8")
print("Partidas restantes após cada filtro:")
print(pd.Series(funil, name="partidas").to_string())
print(f"\n{arquivo}: {metadados['n_jogos']:,} partidas, {metadados['n_linhas']:,} linhas")

Partidas restantes após cada filtro:
todas as partidas do arquivo                  73279
temporada regular e Copa NBA (sem a final)    66834
temporadas de 2005 a 2026                     26494
sem as temporadas de 2020 e 2021              24355
após as verificações de integridade           24350

dados\processados\partidas_times.parquet: 24,350 partidas, 48,700 linhas
